In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. Basic Setup
n_products = 10
n_regions = 5
start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)
date_range = pd.date_range(start_date, end_date)

# 2. MMM Core Logic Functions
def apply_adstock(spend, decay=0.5):
    adstock = np.zeros_like(spend)
    for t in range(1, len(spend)):
        adstock[t] = spend[t] + decay * adstock[t-1]
    return adstock

def apply_saturation(x, alpha=0.0008):
    return 1 - np.exp(-alpha * x)

# 3. Generating Enterprise Data (Targeting 100,000+ Rows)
data_list = []

for prod in range(n_products):
    for reg in range(n_regions):
        product_name = f"Product_{chr(65+prod)}"
        region_name = f"Region_{reg+1}"
        
        # Base Sales logic for variety
        base_sales = np.random.uniform(800, 2000)
        
        # Simulating Marketing Spend Patterns
        google_spend = np.random.gamma(2, 600, len(date_range))
        meta_spend = np.random.gamma(2, 450, len(date_range))
        # TV spend is intermittent (Campaign based)
        tv_spend = np.where(np.random.rand(len(date_range)) > 0.92, np.random.uniform(8000, 20000), 0)
        
        # Applying Adstock (Memory Effect)
        g_adstock = apply_adstock(google_spend, 0.5)
        m_adstock = apply_adstock(meta_spend, 0.3)
        tv_adstock = apply_adstock(tv_spend, 0.7)
        
        # Market Growth & Seasonality Trends
        trend = np.linspace(1, 1.8, len(date_range)) 
        seasonality = 1 + 0.25 * np.sin(2 * np.pi * date_range.dayofyear / 365.25)
        
        # Calculating Final Sales
        sales = (base_sales + 
                 0.6 * apply_saturation(g_adstock) * 1200 + 
                 0.5 * apply_saturation(m_adstock) * 900 + 
                 0.9 * apply_saturation(tv_adstock) * 2500) * trend * seasonality
        
        # Adding some 'Real World' random noise
        sales += np.random.normal(0, 40, len(date_range))
        
        df_temp = pd.DataFrame({
            'Date': date_range,
            'Product': product_name,
            'Region': region_name,
            'Google_Ads_Spend': google_spend.round(2),
            'Meta_Ads_Spend': meta_spend.round(2),
            'TV_Spend': tv_spend.round(2),
            'Sales': np.maximum(sales, 0).round(2)
        })
        data_list.append(df_temp)

# 4. Final Compilation
final_df = pd.concat(data_list)
final_df.to_csv('enterprise_mmm_data.csv', index=False)
print(f"🔥 Success Nanba! Generated {len(final_df)} rows.")
print("File saved as: enterprise_mmm_data.csv")

🔥 Success Nanba! Generated 109600 rows.
File saved as: enterprise_mmm_data.csv
